In [24]:
import numpy as np
import json
import random
from collections import deque
from pathlib import Path

In [25]:
GRID_SIZE = 5
NUM_GRIDS = 100

In [26]:
def is_goal_reachable(obstacles):
    start = (0, 0) # top-left
    goal = (4, 4)  # bottom-right

    queue = deque([start])
    visited = {start}

    directions = [
        (-1, 0),  # North
        (1, 0),   # South
        (0, -1),  # West
        (0, 1)    # East
    ]

    while queue:
        r, c = queue.popleft()

        if (r, c) == goal:
            return True

        for dr, dc in directions:
            nr, nc = r + dr, c + dc
            next_pos = (nr, nc)

            if (
                0 <= nr < GRID_SIZE
                and 0 <= nc < GRID_SIZE
                and next_pos not in obstacles
                and next_pos not in visited
            ):
                visited.add(next_pos)
                queue.append(next_pos)

    return False


In [34]:
def generate_grids(num_grids, grid_size):
    random.seed(42)
    np.random.seed(42)

    grids = []
    seen = set()

    # Exclude start and goal from obstacle candidates
    open_positions = [
        (r, c)
        for r in range(grid_size)
        for c in range(grid_size)
        if (r, c) not in [(0, 0), (4, 4)]
    ]

    while len(grids) < num_grids:
        num_obstacles = random.randint(4, 6)
        obstacles = frozenset(
            random.sample(open_positions, num_obstacles)
        )

        # Unique obstacle configuration
        if obstacles in seen:
            continue

        # Only retain grids with reachable goals
        if is_goal_reachable(obstacles):
          seen.add(obstacles)

          grid = np.full((grid_size, grid_size), '.')

          grid[0, 0] = 'S'  # Start
          grid[4, 4] = 'G'  # Goal

          for r, c in obstacles:
              grid[r, c] = 'B'

          grids.append({
            'id': len(grids),
            'grid': grid.tolist(),
            'start': (0, 0),
            'goal': (4, 4)
          })

    return grids

In [35]:
grids = generate_grids(NUM_GRIDS, GRID_SIZE)

print("Number of grids:", len(grids))

Number of grids: 100


In [36]:
grids[60]['grid']

[['S', '.', '.', '.', '.'],
 ['.', '.', '.', '.', 'B'],
 ['.', '.', '.', '.', '.'],
 ['.', '.', 'B', '.', '.'],
 ['.', 'B', 'B', '.', 'G']]

In [37]:
print("Current directory:", Path.cwd())

with open("data/grids.json", "w") as f:
    json.dump(grids, f, indent=2)

Current directory: /home/sm/projects/llm-grid-solver
